# Azure OpenAI v1 Routing

## What This Sample Does

Learn how APIM chooses a model-specific backend pool for both **Chat Completions** and **Responses**. Four APIs isolate the routing choices: body model, header-first model, subscription-based priority, and a recommended combination of header-first selection with subscription priority.

Follow the six sections in order: **configure -> deploy -> successful requests -> local rejections -> upstream rejections -> telemetry**. Each section begins with a one-sentence recap of where we are and what comes next, followed by its purpose and expected outcome. Resource and probe counts below assume the default two-model configuration.

Three Microsoft Foundry accounts host identical PAYG deployment names. The labels `global-standard`, `ptu` and `consumption` simulate routing preferences only; this notebook does not purchase PTU capacity.

See [README.md](README.md) for prerequisites, costs and live acceptance gates; the optional [full topology](../../assets/diagrams/Azure-OpenAI-v1-Routing.svg) and [Draw.io source](../../assets/diagrams/Azure-OpenAI-v1-Routing.drawio) provide pool membership detail.

### 🛠️ Initialize Notebook Variables

Nothing has been deployed or tested yet; next, we will prepare the model, API and request definitions locally.

**Purpose:** Choose the target infrastructure and PAYG models, then describe the four routing scenarios and both v1 operations before making any Azure calls.

**Expected outcome:** Validated local model/API declarations, example request bodies and Bicep parameters are ready for deployment; invalid configuration stops here without creating resources.

**Only modify entries under _USER CONFIGURATION_.**

- `index` selects the infrastructure resource group. This sample uses the fixed `aoai-v1-routing-1` prefix for its APIs, fragments, pools and subscriptions.
- The default models are `gpt-5-mini` and `gpt-5-nano`, deployed with identical names on all three accounts. Only PAYG `GlobalStandard` or `Standard` is allowed.
- Each API (`body`, `header`, `subscription`, `recommended`) exposes `POST /chat/completions` and `POST /responses` under `/aoai-v1-routing-1/<scenario>/openai/v1`.
- Chat Completions examples use `messages`; Responses examples use `input` and `store: false`. Neither body is converted into the other format.

In [ ]:
# APIM Samples imports
import routing_helpers
import utils
from ai_models import ApprovedModel
from apimtesting import ApimTesting
from apimtypes import API, APIOperation, APIM_SKU, HTTP_VERB, INFRASTRUCTURE, Region
from azure_resources import get_infra_rg_name
from notebook_autoreload import configure_notebook_autoreload

configure_notebook_autoreload()

# ------------------------------
#    USER CONFIGURATION
# ------------------------------

rg_location     = Region.EAST_US_2
index           = 4
apim_sku        = APIM_SKU.BASICV2
deployment      = INFRASTRUCTURE.SIMPLE_APIM
sample_models   = (ApprovedModel.GPT_5_MINI, ApprovedModel.GPT_5_NANO)
model_sku       = 'GlobalStandard'  # Standard also permitted; never provisioned throughput.
model_capacity  = 10  # Three accounts times every model; quota and model conversion apply.
tags            = ['aoai-v1-routing', 'ai-scenarios']

# ------------------------------
#    SYSTEM CONFIGURATION
# ------------------------------

sample_folder = 'aoai-v1-routing'
sample_name = 'aoai-v1-routing-1'
supported_infras = [
    INFRASTRUCTURE.SIMPLE_APIM,
    INFRASTRUCTURE.APIM_ACA,
    INFRASTRUCTURE.APPGW_APIM,
    INFRASTRUCTURE.APPGW_APIM_PE,
    INFRASTRUCTURE.AFD_APIM_PE,
]
model_configuration = routing_helpers.build_models(sample_name, sample_models, model_sku, model_capacity)
rg_name = get_infra_rg_name(deployment, index)
nb_helper = utils.NotebookHelper(
    sample_folder, rg_name, rg_location, deployment, supported_infras,
    index = index, apim_sku = apim_sku,
)

# Each endpoint has its own XML policy; this order matches the Bicep declarations.
scenario_names = ('body', 'header', 'subscription', 'recommended')
paths = {scenario: f'{sample_name}/{scenario}/openai/v1' for scenario in scenario_names}
operations = [
    APIOperation(
        'chat-completions', 'v1 Chat Completions', '/chat/completions', HTTP_VERB.POST,
        'Forward the unchanged Chat Completions body to a native model-specific pool.',
        policyXml = '',  # API scope owns routing and error handling.
    ),
    APIOperation(
        'responses', 'v1 Responses', '/responses', HTTP_VERB.POST,
        'Forward the unchanged Responses body to the same native model-specific pools.',
        policyXml = '',
    ),
]
request_payloads = {
    'chat/completions': {
        'messages': [{'role': 'user', 'content': 'Reply with a short greeting.'}],
        'max_completion_tokens': 1024,
        'reasoning_effort': 'low',
        'stream': False,
    },
    'responses': {
        'input': 'Reply with a short greeting.',
        'max_output_tokens': 1024,
        'reasoning': {'effort': 'low'},
        'stream': False,
        'store': False,  # Native pool selection does not provide conversation-state affinity.
    },
}
apis = [
    API(
        f'{sample_name}-{scenario}', f'Azure OpenAI v1 Routing - {scenario.title()}', paths[scenario],
        f'Native v1 Chat Completions and Responses routing: {scenario}.', operations = operations, tags = tags,
        subscriptionRequired = True,
    )
    for scenario in scenario_names
]
bicep_parameters = {
    'location': {'value': rg_location},
    'models': {'value': model_configuration},
    'modelSku': {'value': model_sku},
    'apis': {'value': [api.to_dict() for api in apis]},
}
verification = ApimTesting('Azure OpenAI v1 Routing', sample_name, deployment)
results = []

### 🚀 Deploy Microsoft Foundry Resources and Native Pools

The deployment definitions are ready locally; next, we will deploy the shared routing resources and retrieve the context needed to call them.

**Purpose:** Create one model-specific backend topology that all four APIs and both v1 operations reuse, retaining the infrastructure's APIM identity and monitoring logger.

**Expected outcome:** With the default two models, deployment provides three Foundry accounts, six model deployments, six concrete backends, four native pools and four APIs with two POST operations each; gateway context and API-scoped keys are available for probes without printing secrets.

`NotebookHelper.deploy_sample()` validates the infrastructure and offers creation or selection when needed. It deploys the six policy fragments before their API references. APIM forwards to the matched v1 operation using managed identity with the Cognitive Services OpenAI User role.

Each model has `standard` and `priority` pools sharing the same concrete backends and circuit breakers. In account order `global-standard / ptu / consumption`:

- Standard uses priorities `[1, 2, 1]` and weights `[3, 1, 1]`.
- Priority uses priorities `[2, 1, 1]` and weights `[1, 3, 1]`.

Lower numeric priority wins; weights apply within the eligible priority group. One backend is selected per request. The `subscription` and `recommended` APIs have separate priority keys that are not interchangeable.

Deployment success does not yet prove inference or RBAC propagation. Sample diagnostics do not capture bodies or headers. Existing deterministic account names are retained for in-place upgrades; read the README upgrade guidance before redeploying an older sample.

In [ ]:
results = []
verification = ApimTesting('Azure OpenAI v1 Routing', sample_name, deployment)
output = nb_helper.deploy_sample(bicep_parameters)
context = nb_helper.get_deployment_context(output)
keys = {scenario: routing_helpers.subscription_key(context.apis, api.name) for scenario, api in zip(scenario_names, apis, strict = True)}
priority_key = output.get('prioritySubscriptionPrimaryKey', suppress_logging = True)
assert isinstance(priority_key, str) and priority_key
recommended_priority_key = output.get('recommendedPrioritySubscriptionPrimaryKey', suppress_logging = True)
assert isinstance(recommended_priority_key, str) and recommended_priority_key
assert output.get('standardSubscriptionName', suppress_logging = True) == f'api-{apis[2].name}'
assert output.get('prioritySubscriptionName', suppress_logging = True) == f'{sample_name}-priority'
assert output.get('recommendedStandardSubscriptionName', suppress_logging = True) == f'api-{apis[3].name}'
assert output.get('recommendedPrioritySubscriptionName', suppress_logging = True) == f'{sample_name}-recommended-priority'

### 🧪 Verify Both v1 Pathways and Models Across All Routing Scenarios

The shared routing resources are deployed; next, we will send valid requests through every scenario and verify the selected pools and generated results.

**Purpose:** Demonstrate how model selection and validated subscription priority affect routing on both Chat Completions and Responses without changing the caller's body.

**Expected outcome:** With two models, two operations and eight comparisons, all 32 requests return HTTP 200, generated text in the correct response format, the exact expected pool and a correlation ID.

The eight comparisons per model and operation cover:

- `body`: select the body model and the standard family.
- `header`: select the known `x-ai-model` header and the standard family, without reading the body.
- `subscription`: send identical paths and bodies with standard versus priority keys. A caller's `x-priority` header cannot grant priority.
- `recommended`: combine header-first selection with standard/priority keys, then exercise empty-header and absent-header body fallback.

**Example:** A `recommended` request selecting `gpt-5-mini` with its priority subscription must report `aoai-v1-routing-1-gpt-5-mini-priority-pool`. Changing only to the standard key must select the corresponding `standard-pool`. The body still names `gpt-5-mini`; the header does not rewrite it.

Chat Completions must contain text in `choices`; Responses requires completed assistant text in `output`. These small probes do not prove a weighted distribution or exercise live streaming.

**Why Responses uses `store: false`:** Pools provide no account affinity for `previous_response_id` or stored-response retrieval. Supply conversation context in `input`. All paths make one backend attempt with no retries; breakers affect subsequent requests. If a probe fails, use its sanitized error identity and request ID rather than hiding the failure with retries.

In [ ]:
for model in model_configuration:
    selector = model['deploymentName']
    scenarios = [
        ('body', 'standard', keys['body'], {}),
        ('header', 'standard', keys['header'], {'x-ai-model': selector}),
        ('subscription', 'standard', keys['subscription'], {'x-priority': 'priority'}),
        ('subscription', 'priority', priority_key, {}),
        ('recommended', 'standard', keys['recommended'], {'x-ai-model': selector, 'x-priority': 'priority'}),
        ('recommended', 'priority', recommended_priority_key, {'x-ai-model': selector, 'x-priority': 'priority'}),
        ('recommended', 'standard', keys['recommended'], {'x-ai-model': ''}),
        ('recommended', 'priority', recommended_priority_key, {}),
    ]
    for pathway, request_payload in request_payloads.items():
        payload = {'model': selector, **request_payload}
        for scenario, family, key, headers in scenarios:
            result = routing_helpers.probe(
                lambda: nb_helper.create_apim_requests(context.apim_gateway_url, key),
                f'/{paths[scenario]}/{pathway}', payload, headers = headers,
            )
            expected_pool = f'{sample_name}-{selector}-{family}-pool'
            label = f'{pathway}/{scenario}/{family}/{selector}'
            assert result.status == 200 and result.completion, f'{label}: {result}'
            assert verification.verify(result.pool, expected_pool, f'{label} pool')
            results.append(result)

### 🧪 Verify Missing Convention Pools, Local Rejections and Header Fallback

Valid requests have demonstrated successful routing; next, we will check how APIM rejects invalid selectors and unavailable routes while preserving valid header fallback.

**Purpose:** Distinguish local input validation from native pool-resolution failure and prove that only an empty routing header falls back to the body.

**Expected outcome:** All 60 rejection/fallback probes match their expected status and error identity; rejected requests have no selected-pool evidence, while successful empty-header fallbacks identify the standard pool.

Six rejection cases run on every API and both operations:

- `unknown-model` has valid syntax but no deployed pool. The expected resolver result is 404 `model_not_available`.
- A missing body model is rejected locally with 404 `ModelNotAvailable`.
- A non-object body, non-boolean `stream` or numeric model is rejected locally with 400 `InvalidRequest`.

It then tests `header` and `recommended` separately: an **empty** `x-ai-model` falls back to the valid body and succeeds; an unknown or invalid **nonempty** header must not fall back, even though the body names a valid model. The client rejects whitespace-only headers before sending, so these live probes use an empty string.

Both missing-model codes use **The specified model is not available for use**. These errors do not prove that an upstream model deployment is missing. Local early returns emit a trace before stopping because outbound policies will not run.

**Important live gate:** The missing-pool 404 requires `resolve-model-pool` attribution in inbound while `routePhase` is `resolve`. If the gateway defers rejection or reports a different attribution, the assertion intentionally fails. Do not relabel a generic gateway failure to pass the test. This section does not delete pools or check a model membership table.

In [ ]:
for pathway, request_payload in request_payloads.items():
    valid_payload = {'model': model_configuration[0]['deploymentName'], **request_payload}
    negative_cases = [
        ({**valid_payload, 'model': 'unknown-model'}, 404, 'model_not_available'),
        (request_payload, 404, 'ModelNotAvailable'),
        ('not-an-object', 400, 'InvalidRequest'),
        ('[]', 400, 'InvalidRequest'),
        ({**valid_payload, 'stream': 'false'}, 400, 'InvalidRequest'),
        ({**valid_payload, 'model': 123}, 400, 'InvalidRequest'),
    ]
    for scenario in scenario_names:
        key = keys[scenario]
        for payload, expected_status, expected_code in negative_cases:
            result = routing_helpers.probe(
                lambda: nb_helper.create_apim_requests(context.apim_gateway_url, key),
                f'/{paths[scenario]}/{pathway}', payload,
            )
            assert result.status == expected_status and result.error_code == expected_code
            if expected_code in ('ModelNotAvailable', 'model_not_available'):
                assert result.error_message == 'The specified model is not available for use'
            assert not result.pool  # No concrete backend was successfully selected.
            results.append(result)

    for scenario in ('header', 'recommended'):
        for header, expected_code in [('', None), ('unknown-model', 'model_not_available'), ('../invalid', 'ModelNotAvailable')]:
            result = routing_helpers.probe(
                lambda: nb_helper.create_apim_requests(context.apim_gateway_url, keys[scenario]),
                f'/{paths[scenario]}/{pathway}', valid_payload, headers = {'x-ai-model': header},
            )
            if expected_code:
                assert result.status == 404 and result.error_code == expected_code
                assert result.error_message == 'The specified model is not available for use'
            else:
                assert result.status == 200 and result.completion
                assert result.pool == f'{sample_name}-{valid_payload["model"]}-standard-pool'
            results.append(result)

### 🧪 Verify Real Upstream Mismatch Errors Without Body Rewriting

Local rejection and fallback behavior has been checked; next, we will select known pools but send deliberately invalid bodies to test upstream rejection.

**Purpose:** Prove that header-first routing leaves the body unread and unchanged, and that APIM distinguishes a successful route selection from a failed upstream request.

**Expected outcome:** With two models, all 12 probes return the expected upstream-specific errors with nonempty pool evidence, the original HTTP status and the safe message **The upstream service rejected the request.**

Both header-first APIs and both operations are tested:

1. For each configured model, select its known pool with `x-ai-model` but send `model = 'not-deployed-on-any-account'` in the body. The caller deliberately creates a mismatch; APIM does not repair it.
1. Send a non-object body with a known header. It must reach upstream validation, proving that header-first routing bypassed local body parsing.

The model mismatch requires HTTP 404 with `DeploymentNotFound` or `model_not_found`; the non-object body requires HTTP 400 with a sanitized upstream error identity. No arbitrary upstream message is exposed.

**Why this matters:** A bare 404 cannot distinguish an unavailable APIM route from an Azure deployment-not-found response. Check the error identity, safe message and selected pool together. These assertions are live acceptance gates: if Azure changes its error contract, investigate the failure rather than accepting any 404.

In [ ]:
for pathway, request_payload in request_payloads.items():
    for scenario in ('header', 'recommended'):
        for model in model_configuration:
            selector = model['deploymentName']
            mismatch = {**request_payload, 'model': 'not-deployed-on-any-account'}
            result = routing_helpers.probe(
                lambda: nb_helper.create_apim_requests(context.apim_gateway_url, keys[scenario]),
                f'/{paths[scenario]}/{pathway}', mismatch, headers = {'x-ai-model': selector},
            )
            assert result.status == 404
            assert result.error_code in ('DeploymentNotFound', 'model_not_found'), 'Expected a real Azure deployment-not-found error, not a local model rejection.'
            assert result.error_message == 'The upstream service rejected the request.'
            assert result.pool == f'{sample_name}-{selector}-standard-pool'
            results.append(result)

        result = routing_helpers.probe(
            lambda: nb_helper.create_apim_requests(context.apim_gateway_url, keys[scenario]),
            f'/{paths[scenario]}/{pathway}', 'not-an-object',
            headers = {'x-ai-model': model_configuration[0]['deploymentName']},
        )
        assert result.status == 400 and result.error_code in ('InvalidRequestError', 'BadRequest', 'UpstreamError')
        assert result.error_message == 'The upstream service rejected the request.'
        assert result.pool == f'{sample_name}-{model_configuration[0]["deploymentName"]}-standard-pool'
        results.append(result)

### 📊 Report Sanitized Evidence and Correlate Error Traces

The success and rejection probes have collected their results; next, we will summarize that evidence and correlate error request IDs with Application Insights traces.

**Purpose:** Review the sanitized outcomes and independently verify error observability without exposing prompts, keys or arbitrary upstream content.

**Expected outcome:** With the default two models and all prior cells complete, the report lists 104 results and the shared verification summary has no failures; trace coverage is confirmed separately by manually matching error request IDs in Logs.

The report contains 32 success comparisons, 60 rejection/fallback probes and 12 upstream-error probes. The next cell prints sanitized `ProbeResult` fields and checks the verification summary; it **does not execute a Logs query or automatically prove trace ingestion**.

**Read the evidence:** `status` and `error_code` distinguish the outcome; `pool` distinguishes an upstream-selected route from an earlier rejection; `request_id` is the correlation value from `x-routing-request-id`. A failed success probe includes these safe fields in its assertion. Policy failures additionally trace a bounded machine-readable reason, never arbitrary `LastError.Message`.

**Verify traces manually:**

1. Copy an error result's `request_id` and open [routing-errors.kql](queries/routing-errors.kql).
1. Set the query's `requestId` binding to that value and run it in Application Insights Logs, allowing time for ingestion.
1. Confirm matching `RequestId`, `Scenario`, `ErrorKind` and `Status`; inspect `Reason` for policy failures. The query also joins request telemetry without projecting URLs or bodies.

**Coverage boundary:** Invalid subscription keys and unmatched operations fail before API policies run, so they cannot emit these sample traces. Use gateway/request diagnostics for them. Do not infer missing-pool reasons from arbitrary error text.

This section performs no cleanup. Follow the [README cleanup guidance](README.md#clean-up) to remove only sample-owned resources; incremental deployments do not delete removed models, and no destructive pool-deletion probe runs here.

In [ ]:
assert results, 'Run the live verification cells first.'
for result in results:
    print(result)
assert verification.tests_failed == 0
verification.print_summary()
print('Live gates: verify scoped missing-route 404 assertions and correlate error request IDs in Application Insights, including bounded original Reason for policy failures.')